⚠️ **DEPRECATED — Phase 7c.2 architecture supersedes this notebook**

This notebook produces the per-session spatial NPZ cache, which was eliminated under Phase 7c.2 Addendum #3 Q2.B (Interpretation 1) in favor of compute-on-demand spatial figures via `src/stroke_orofacial/dlc_spatial/render_on_demand.py::compute_animal_aggregated_histogram`.

The notebook is preserved here for historical reference. Re-running it would write to `pipeline_outputs/dlc_analysis/spatial_<scope>/...` paths that were cleaned up 2026-05-23.

The PathResolver method `spatial_tone_aligned_npz()` in `paths.py` remains dormant (opt-in cache write only via `tone_aligned.py::compute_tone_aligned(..., emit_cache=True)`).

For current spatial analyses, use:
- `dlc_spatial.render_on_demand` for ad-hoc histograms
- `scripts/phase_9_stats/materialize_spatial.py` for stats inputs
- Existing Phase 7c.2 SVGs under `dlc_kinematics/tone_aligned_heatmaps/` and `dlc_kinematics/whole_session_heatmaps/` for the curated figure set

# DLC Spatial Analysis (Whole Session)

Pixel-bin occupancy maps, 2D heatmaps, and difference maps for DLC bodypart tracking across whole sessions.

Outputs go to `stroke_orofacial_pipeline_outputs/dlc_analysis/spatial_whole_session/` on MICROSCOPE.

**Phase 6.7 update:** orchestration now flows through `SessionManifest`. Cell 3 builds one manifest per (animal, date); cell 4 iterates and consumes `manifest.dlc_h5_files` directly to locate the centered parquet (no more re-discovery via `discover_dlc_filtered_files`). Phase 5 is DLC-required — manifests with no DLC files raise FileNotFoundError, surfaced in the per-(bodypart, mode) error log.

**Per Priya 2026-04-20:** every fine-grid heatmap is now also saved as `heatmap_{date}.npz` alongside the PNG (via `save_heatmap_npz`). Future figure tweaks (different colormaps, axis limits, normalization) and Phase 9 fine-resolution stats can re-render from the NPZ without recomputing from parquet. The NPZ resolution is configurable via `HEATMAP_NBINS` below.

In [ ]:
# ============================================================
# USER INPUTS — edit variables below, then run all cells.
# ============================================================

ANIMALS = ["PS46", "PS47", "PS48", "PS49", "PS50", "PS51", "PS54", "PS55"]

# Dates to process per animal. Set to None for all dates in session range.
DATES = None

# Bodyparts to analyze
BODYPARTS = ["nose", "jaw", "tongue"]

# Centering modes (each bodypart analyzed under each mode)
CENTER_MODES = ["nose", "spout_midpoint", "eye_midpoint"]

# Fine-grid heatmap NPZ resolution. 100 matches plot_heatmap's internal
# default that produced the existing PNGs (per DECISIONS.md 2026-04-19).
# Bump higher to explore alternative resolutions; the PNG itself remains
# at the same resolution as before.
HEATMAP_NBINS = 100

# Overwrite mode: "skip" (default) or "force"
OVERWRITE = "skip"

In [ ]:
import datetime
import logging

import numpy as np

from stroke_orofacial.animals import load_animals
from stroke_orofacial.config_loader import load_defaults
from stroke_orofacial.dlc_io.parquet import _param_sig, read_centered_parquet
from stroke_orofacial.dlc_spatial.heatmaps import (
    plot_heatmap,
    save_heatmap_npz,
    save_heatmap_png,
)
from stroke_orofacial.dlc_spatial.pixel_bins import build_stats_grid
from stroke_orofacial.manifest import build_manifest
from stroke_orofacial.paths import PathResolver

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

resolver = PathResolver()
animals_config = load_animals()
defaults = load_defaults()
spatial_config = defaults["dlc_spatial"]
centering_config = defaults.get("dlc_centering", {})
output_root = resolver.pipeline_outputs("dlc_analysis") / "spatial_whole_session"
lik_thr_centering = centering_config.get("likelihood_threshold", 0.7)

In [ ]:
# ============================================================
# Build a SessionManifest per (animal, date). Surfaces DLC h5
# discovery so downstream parquet locating uses manifest.dlc_h5_files
# instead of re-running discover_dlc_filtered_files.
# ============================================================

manifests = []  # list of (animal_id, date, manifest)

for animal_id in ANIMALS:
    info = animals_config[animal_id]

    if DATES is not None:
        dates = DATES
    else:
        n_days = (info.session_end - info.session_start).days + 1
        dates = [info.session_start + datetime.timedelta(days=i) for i in range(n_days)]

    for date in dates:
        manifest = build_manifest(animal_id, date, resolver, info)
        manifests.append((animal_id, date, manifest))

n_with_dlc = sum(1 for _, _, m in manifests if m.dlc_h5_files)
print(f"Built {len(manifests)} manifests; {n_with_dlc} have DLC files (will process).")

In [ ]:
# ============================================================
# For each (manifest, center_mode, bodypart): locate the centered
# parquet via manifest.dlc_h5_files[0], render heatmap PNG, save
# fine-grid NPZ (HEATMAP_NBINS), and save coarse stats grid NPZ.
# ============================================================

for center_mode in CENTER_MODES:
    sig = _param_sig(center_mode, lik_thr_centering)
    print(f"\n{'='*60}\ncenter_mode={center_mode}\n{'='*60}")

    for bodypart in BODYPARTS:
        n_ok, n_skip, n_err = 0, 0, 0

        for animal_id, date, manifest in manifests:
            date_str = date.strftime("%Y%m%d")

            if not manifest.dlc_h5_files:
                n_err += 1
                continue

            parquet_dir = (
                resolver.pipeline_outputs("dlc_analysis")
                / f"centered_{center_mode}" / animal_id
            )
            parquet_name = f"{manifest.dlc_h5_files[0].name}.intermediate.{sig}.parquet"
            parquet_path = parquet_dir / parquet_name
            if not parquet_path.exists():
                n_err += 1
                continue

            out_dir = output_root / f"{bodypart}_{center_mode}" / animal_id
            heatmap_path = out_dir / f"heatmap_{date_str}.png"
            heatmap_npz_path = out_dir / f"heatmap_{date_str}.npz"

            if OVERWRITE == "skip" and heatmap_path.exists():
                n_skip += 1
                continue

            try:
                df = read_centered_parquet(parquet_path)

                # Heatmap PNG (PNG content unchanged from pre-6.7d)
                fig = plot_heatmap(
                    df, bodypart, spatial_config,
                    center_mode=center_mode,
                    title=f"{animal_id} {bodypart} {date_str} ({center_mode})",
                )
                save_heatmap_png(fig, heatmap_path, dpi=spatial_config.get("save_dpi", 200))

                # Fine-grid heatmap NPZ alongside the PNG (Phase 6.7d addition)
                save_heatmap_npz(
                    df, bodypart, spatial_config, heatmap_npz_path,
                    center_mode=center_mode, nbins=HEATMAP_NBINS,
                )

                # Coarse stats grid NPZ (unchanged)
                grid = build_stats_grid(
                    df, bodypart, spatial_config, center_mode=center_mode,
                )
                if grid is not None:
                    npz_path = out_dir / f"stats_grid_{date_str}.npz"
                    npz_path.parent.mkdir(parents=True, exist_ok=True)
                    np.savez(str(npz_path), **grid)

                n_ok += 1
            except Exception as e:
                n_err += 1
                logging.warning(
                    "%s/%s/%s/%s: %s", animal_id, date_str, bodypart, center_mode, e,
                )

        print(f"  {bodypart}: ok={n_ok} skip={n_skip} err={n_err}")